In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Lambda
from tensorflow.keras.models import Model

# 1. Load and preprocess dataset
path = "Datasets/Object Detection/fashion mnist"

train_ds = tf.keras.utils.image_dataset_from_directory(
    path, validation_split=0.2, subset="training",
    seed=42, image_size=(224, 224), batch_size=32,
    color_mode="rgb"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    path, validation_split=0.2, subset="validation",
    seed=42, image_size=(224, 224), batch_size=32,
    color_mode="rgb"
)

classes = train_ds.class_names

# 2. Load pretrained VGG16 model
base = VGG16(weights=None, include_top=False,
             input_shape=(224, 224, 3))

base.load_weights("Datasets/Object Detection/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5")

base.trainable = False

# 3. Add custom classification layers
inp = Input(shape=(224, 224, 3))
x = Lambda(tf.keras.applications.vgg16.preprocess_input)(inp)
x = base(x)
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
out = Dense(len(classes), activation="softmax")(x)

model = Model(inp, out)
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# 4. Train the model
model.fit(train_ds, validation_data=val_ds, epochs=5)

# 5. Evaluate the model
loss, accuracy = model.evaluate(val_ds)
print("Loss:", loss)
print("Accuracy:", accuracy)

# 6. Fine-tune the model
base.trainable = True

for layer in base.layers[:-20]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(0.00001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.fit(train_ds, validation_data=val_ds, epochs=2)

# 7. Visualize prediction
images, labels = next(iter(val_ds))
pred = model.predict(images[:1])
p = pred[0].argmax()

plt.imshow(images[0].numpy().astype("uint8"))
plt.title(f"Actual: {classes[labels[0]]}, Predicted: {classes[p]}")
plt.axis("off")
plt.show()